## 📚 Índice do Notebook

1. [📊 Automatic IDF Graphs – Análise Exploratória](#-automatic-idf-graphs--análise-exploratória)
2. [🧠 O que é esta biblioteca?](#-o-que-é-esta-biblioteca)
3. [❓ Qual problema ela resolve?](#-qual-problema-ela-resolve)
4. [🛠️ O que este notebook faz?](#-o-que-este-notebook-faz)
5. [⚙️ Instalação e configuração do ambiente](#-instalação-e-configuração-do-ambiente)
6. [📌📆 Requisição de dados brutos de precipitação](#-requisição-de-dados-brutos-de-precipitação)
   - [🛠️ Script para baixar dados (INMET, CEMADEN e CLIMBra)](#-script-para-baixar-dados-inmet-e-cemaden)
   - [✅ Passos para rodar o script](#-passos-para-rodar-o-script)
   - [🖥️ Exemplo de uso](#-exemplo-de-uso)
   - [📌 Dúvidas e suporte](#-dúvidas-e-suporte)
7. [📋 Preenchimento de Falhas nos Dados](#-preenchimento-de-falhas-nos-dados)
   - [🗓️ Interpolação Sazonal](#-interpolação-sazonal)
   - [🛰️ Uso de Dataset Auxiliar](#-uso-de-dataset-auxiliar)
   - [🧠 Quando usar o quê?](#-quando-usar-o-quê)

# ☔💧 Automatic IDF Graphs – Introdução e Aquisição dos Dados

Bem-vindo ao notebook de exploração da biblioteca **Automatic-IDF-Graphs** 🚀

<br>

## 🧠 O que é esta biblioteca?

A `idf_analysis` é uma biblioteca Python voltada à **geração automática de curvas IDF (Intensidade-Duração-Frequência)** a partir de dados históricos ou projeções climáticas. Ela auxilia na construção de curvas que são fundamentais para:

- Projetos de drenagem urbana e rural 🌧️
- Dimensionamento de obras hidráulicas 🏗️
- Estudos de impacto climático 📉

<br>

## ❓ Qual problema ela resolve?

Tradicionalmente, gerar curvas IDF demanda muito tempo e envolvimento manual com dados pluviométricos. A `idf_analysis` automatiza:

- 📥 Coleta e tratamento dos dados
- 📈 Análise estatística e ajuste de distribuições
- 📐 Geração das curvas e tabelas
- 📊 Visualização gráfica

<br>

## 🛠️ O que este notebook faz?

Neste notebook, vamos:

- 📖 Apresentar a biblioteca e explicar a necessidade da aquisição de dados.
- ☁️ Demonstrar como requisitar e carregar dados de diferentes fontes (INMET, CEMADEN, CLIMBra).
- 📊 Gerar um sumário estatístico para entender a abrangência de cada conjunto de dados.
- 💧 Detalhar e aplicar métodos para o preenchimento de falhas em séries temporais.


> 💡 *Este é um ponto de partida para entender e testar a biblioteca antes de aplicá-la em casos reais.*

#### Seguiremos um passo a passo claro, desde instalações até a visualização dos dados.

<br>

---


## ⚙️ Instalação e configuração do ambiente

Instalamos a biblioteca diretamente do repositório GitHub e incluímos a pasta `src/` no `sys.path` para facilitar a importação dos módulos locais.


In [4]:
import sys
import warnings
warnings.filterwarnings('ignore')

print("Instalando a biblioteca...\n")
%pip install --quiet -e ..
sys.path.append('../src')

print("\nInstalação concluída!")

Instalando a biblioteca...

Note: you may need to restart the kernel to use updated packages.

Instalação concluída!


---
## 📌📆 Requisitando dados brutos de precipitação para a análise.

### 🛠️ Como usar o script interativo para baixar dados (INMET, CEMADEN e CLIMBra)

Este projeto inclui um script interativo para facilitar o download de dados meteorológicos do INMET e CEMADEN, além das projeções históricas e futuras do CLIMBra.

*⚠️ Importante*: As funções de download usam menus interativos via terminal, que **não funcionam dentro de notebooks Jupyter**, especialmente no Windows.

#### ✅ Passos para rodar o script

1.  **Abra um terminal** (Prompt de Comando, PowerShell, terminal do VSCode, etc.)
2.  Navegue até a raiz do projeto (onde está a pasta `scripts/`):
    ```bash
    cd /caminho/para/Automatic-IDF-Graphs
    ```
3.  Execute o script:
    ```bash
    python scripts/download_data.py
    ```
4.  Você verá um menu interativo para escolher entre baixar dados do INMET, do CEMADEN ou do CLIMBra, e depois escolher parâmetros como estado, cidade, estação e período.

<br>

### ↪️ Alternativa: Acesso direto aos dados

Se preferir, você também pode requisitar os dados diretamente dos portais oficiais de cada instituição. Esta pode ser uma boa opção para explorar os dados visualmente antes de baixar ou para downloads mais específicos.

* **CEMADEN:** [https://mapainterativo.cemaden.gov.br](https://mapainterativo.cemaden.gov.br)
* **INMET:** [https://portal.inmet.gov.br](https://portal.inmet.gov.br)
* **CLIMBra:** [https://www.scidb.cn/en/detail?dataSetId=609b7ff93f0d4d1a9ba6eb709027c6ad](https://www.scidb.cn/en/detail?dataSetId=609b7ff93f0d4d1a9ba6eb709027c6ad)

<br>

### 🖥️ Exemplo de uso do script

Após rodar o script, o terminal mostrará algo assim:

```bash
? Qual base de dados deseja baixar? (Use as setas ↑ ↓ para navegar)
❯ CEMADEN
  INMET
  CLIMBra

✅ Token recebido com sucesso!

? Selecione um estado (UF):
❯ SP
  RJ
  MG
  ...
```

<br>

### 📌 Dúvidas e suporte
Se tiver problemas ou sugestões, abra uma issue no GitHub.

---
> ⚠️ *Nesse notebook exploratório foram utilizados cinco bases de dados. Dentre elas, temos duas frentes de análise:
- _Mirante de Santana, São Paulo (INMET, horário, 2014-2024)_
- _Mirante de Santana, São Paulo (CEMADEN, 2014-2024)_

<br>

- _Mirante de Santana, São Paulo (INMET, diário, 1980-2013)_
- _CABra467, São Paulo (CLIMBra, histórico, 1980-2013)_
- _CABra467, São Paulo (CLIMBra, ssp245, 2015-2100)_

<br>

> A primeira será utilizada para a geração de IDF's históricas, com preenchimento de falhas e diversas outras funcionalidades. A segunda será voltada para correção de viés e geração de IDF's de projeção futura.

In [13]:

import pandas as pd

from idf_analysis.data.processing import read_csv, verification, fill_missing_data
from idf_analysis.helpers.notebook import precip_summary
from IPython.display import display

# Lendo os DataFrames obtidos
cemaden_df = read_csv(path='../results/cemaden_ac_santana_sao/cemaden_ac_santana_sao_hourly.csv')
inmet_df = read_csv(path='../results/inmet_sao_paulo_mirante/inmet_sao_paulo_mirante_hourly.csv')
inmet_daily_df = read_csv(path='../results/inmet_daily_sao_paulo_mirante/inmet_daily_sao_paulo_mirante_daily.csv')
climbra_hist_df = read_csv(path='../results/CABra467/historical/1980-2013_daily.csv')
climbra_future_df = read_csv(path='../results/CABra467/ssp245/2015-2100_daily.csv')

summary_hourly = pd.concat([
    precip_summary(inmet_df, "INMET Horário"),
    precip_summary(cemaden_df, "CEMADEN"),
    
])
print("📊 Base de dados para IDF's históricas: ")
display(summary_hourly.style.format(precision=2))

summary_daily = pd.concat([
    precip_summary(inmet_daily_df, "INMET Diário"),
    precip_summary(climbra_hist_df, "CLIMBra Histórico"),
    precip_summary(climbra_future_df, "CLIMBra Futuro"),
])

print("📊 Base de dados para correção de viés e IDF's de projeção futura: ")
display(summary_daily.style.format(precision=2))

📊 Base de dados para IDF's históricas: 


,Total registros,Registros > 0,Primeira data,Última data,Precipitação média (mm),Precipitação máxima (mm)
INMET Horário,96456,7267,2014-01-01 00:00:00,2025-01-01 23:00:00,0.17,77.80
CEMADEN,61329,6038,2014-01-01 00:00:00,2025-01-01 23:00:00,0.12,45.00


📊 Base de dados para correção de viés e IDF's de projeção futura: 


,Total registros,Registros > 0,Primeira data,Última data,Precipitação média (mm),Precipitação máxima (mm)
INMET Diário,12419,4594,1980-01-01 00:00:00,2013-12-31 00:00:00,4.46,151.80
CLIMBra Histórico,12419,5475,1980-01-01 00:00:00,2013-12-31 00:00:00,4.15,128.23
CLIMBra Futuro,31411,13809,2015-01-01 00:00:00,2100-12-31 00:00:00,4.32,149.90


---
## 📋 Preenchimento de Falhas nos Dados

Falhas em séries de precipitação são comuns e devem ser tratadas para garantir análises consistentes. A biblioteca oferece dois métodos principais:

### 1. 🗓️ Interpolação Sazonal
Preenche valores faltantes com base em outros anos no mesmo mês ou estação.  
✅ Ideal para falhas pontuais em séries longas (≥10 anos).  
🎯 Preserva o comportamento sazonal natural da precipitação.

### 2. 🛰️ Uso de Dataset Auxiliar
Usa dados de estações próximas via regressão linear ou múltipla.  
✅ Útil para falhas longas ou estações com boa correlação geográfica.

### 🧠 Quando usar o quê?
- Use **Interpolação Sazonal** para falhas curtas em séries bem consolidadas.  
- Use **Dataset Auxiliar** quando há estações vizinhas confiáveis e falhas extensas.

In [14]:
# Verificando falhas
verification(df=cemaden_df,frequency='hourly')
verification(df=inmet_df,frequency='hourly');

[WARNING] Série incompleta. Períodos faltando: 35127

[OK] Série completa! Nenhum período faltando.



##### 📑 Nesse caso, a *utlização de um dataset auxiliar* é adequada, pois para a fonte do CEMADEN existem muitos dados faltantes.

In [15]:
cemaden_df = fill_missing_data(df_main=cemaden_df,
                               df_secondary=inmet_df,
                               frequency='hourly',)

# Verificando novamente após o preenchimento
verification(df=cemaden_df, frequency='hourly');
display(precip_summary(cemaden_df,'CEMADEN sem falhas').style.format(precision=2))
print("\n✅ Observe que o número de registros aumentou de 61329 para 96456.")

[OK] Série completa! Nenhum período faltando.



,Total registros,Registros > 0,Primeira data,Última data,Precipitação média (mm),Precipitação máxima (mm)
CEMADEN sem falhas,96456,26580,2014-01-01 00:00:00,2025-01-01 23:00:00,0.14,45.00



✅ Observe que o número de registros aumentou de 61329 para 96456.


##### ↪️ Caso queira utilizar a interpolação sazonal (não recomendada para uma quantidade alta de dados faltantes):

In [16]:
cemaden_interpolated_df = read_csv(path='../results/cemaden_ac_santana_sao/cemaden_ac_santana_sao_hourly.csv')

# Verificando falhas antes do preenchimento
verification(df=cemaden_interpolated_df, frequency='hourly');

cemaden_interpolated_df = fill_missing_data(df_main=cemaden_interpolated_df,
                               frequency='hourly',)

# Verificando novamente após o preenchimento
verification(df=cemaden_interpolated_df, frequency='hourly');

[WARNING] Série incompleta. Períodos faltando: 35127

[OK] Série completa! Nenhum período faltando.

